
╔══════════════════════════════════════════════════════════════════════════════╗
║         FEATURE ENGINEERING — POLYMARKET TIMESERIES BUILDER                  ║
║  Run cells top-to-bottom. Only the CONFIG cell requires manual input.        ║
╚══════════════════════════════════════════════════════════════════════════════╝


#### Polymarket — Timeseries Dataset Builder
This notebook:
1. Downloads the **latest SQLite backup** from S3 (skips if already local)
2. Checks if a matching **timeseries CSV** already exists in `./Data/`
3. If not, **pivots the raw markets table** into a wide timeseries (rows = scraped_at, columns = MultiIndex[variable, question])
4. Runs a **sanity check** on the resulting dataframe


In [1]:
# %% ── CELL 1 : CONFIG ────────────────────────────────────────────────────────
import os

BUCKET_NAME     = "thesis-data-ab3rnhard"
S3_PREFIX       = "backups/"
LOCAL_DATA_DIR  = "./Data"
TABLE_NAME      = "markets"
SCRAPED_AT_COL  = "scraped_at"
QUESTION_COL    = "question"

# Columns to DROP before pivoting (identifiers / near-100%-null / pure metadata)
# Add or remove entries here to control which variables end up in the timeseries.
COLS_TO_DROP = [
    "id",                    # row identifier, not a timeseries variable
    "conditionId",           # static identifier
    "slug",                  # static identifier
    "questionID",            # static identifier
    "clobTokenIds",          # static identifier
    "icon", "image",         # URL strings
    "description",           # long text
    "resolutionSource",      # text URL
    # "outcomes",              # JSON text
    # "outcomePrices",         # JSON text (parsed separately if needed)
    "events",                # JSON text
    "umaResolutionStatuses", # JSON text
    "submitted_by",          # text
    "marketMakerAddress",    # 100 % null
    "seriesColor",           # 100 % null
    "liquidityAmm",          # 100 % null
    "oneYearPriceChange",    # 100 % null
    "volume1moAmm",          # 100 % null
    "volume1wkAmm",          # 100 % null
    "volume1yrAmm",          # 100 % null
    "volume24hrAmm",         # 100 % null
    "volumeAmm",             # 100 % null
    "umaResolutionStatus",   # 99 % null
]

# AWS credentials — read from environment variables (same pattern as Health Check)
AWS_ACCESS_KEY_ID     = (os.getenv("AWS_ACCESS_KEY_ID")     or "").strip()
AWS_SECRET_ACCESS_KEY = (os.getenv("AWS_SECRET_ACCESS_KEY") or "").strip()
AWS_REGION            = (os.getenv("AWS_REGION")            or "eu-north-1").strip() or "eu-north-1"

print("✅ Config loaded.")


✅ Config loaded.


In [2]:
# %% ── CELL 2 : DEPENDENCIES ─────────────────────────────────────────────────
try:
    import boto3
except ModuleNotFoundError:
    %pip install boto3 -q
    import boto3

import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from botocore.exceptions import ClientError

print("✅ Dependencies ready.")



✅ Dependencies ready.


In [3]:
# %% ── CELL 3 : FETCH LATEST DB FROM S3 (SKIP IF ALREADY LOCAL) ─────────────
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

missing_vars = [v for v, val in [
    ("AWS_ACCESS_KEY_ID",     AWS_ACCESS_KEY_ID),
    ("AWS_SECRET_ACCESS_KEY", AWS_SECRET_ACCESS_KEY),
] if not val]

if missing_vars:
    raise ValueError(
        "Missing AWS env variable(s): " + ", ".join(missing_vars)
        + ". Set them and restart the kernel."
    )

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION,
)

# ── identify the latest backup on S3 ─────────────────────────────────────────
try:
    response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=S3_PREFIX)
except ClientError as e:
    code = e.response.get("Error", {}).get("Code", "Unknown")
    raise RuntimeError(f"S3 request failed ({code}). Check credentials / bucket.") from e

objects = response.get("Contents", [])
if not objects:
    raise FileNotFoundError(f"No objects found in s3://{BUCKET_NAME}/{S3_PREFIX}")

latest_obj  = sorted(objects, key=lambda x: x["LastModified"], reverse=True)[0]
s3_key      = latest_obj["Key"]
file_date   = latest_obj["LastModified"].strftime("%Y-%m-%d")
local_name  = f"local_database_{file_date}.sqlite"
local_path  = Path(LOCAL_DATA_DIR) / local_name

print(f"Latest S3 object  : {s3_key}")
print(f"Last modified     : {latest_obj['LastModified']}")
print(f"Size              : {latest_obj['Size'] / 1024:.1f} KB")
print(f"Expected local    : {local_path}")

# ── download only if not already present ─────────────────────────────────────
if local_path.exists():
    print(f"\n⏭  DB already present locally — skipping download.")
else:
    print(f"\n⬇  Downloading …")
    s3.download_file(BUCKET_NAME, s3_key, str(local_path))
    print("✅ Download complete.")



Latest S3 object  : backups/db_2026-04-30.sqlite
Last modified     : 2026-04-30 22:10:34+00:00
Size              : 7179532.0 KB
Expected local    : Data\local_database_2026-04-30.sqlite

⏭  DB already present locally — skipping download.


In [4]:
# %% ── CELL 4 : CHECK FOR EXISTING TIMESERIES CSV ────────────────────────────
csv_name = f"timeseries_{file_date}.csv"
csv_path = Path(LOCAL_DATA_DIR) / csv_name

print(f"Looking for CSV: {csv_path}")

if csv_path.exists():
    print("✅ CSV found — loading directly (skipping DB transform).")
    df = pd.read_csv(
        csv_path,
        index_col=0,        # scraped_at is the row index
        header=[0, 1],      # MultiIndex columns: (variable, question)
        parse_dates=True,
    )
    df.index = pd.to_datetime(df.index)
    print(f"   Shape: {df.shape}")
    _csv_loaded = True
else:
    print("⚠️  CSV not found — will build from SQLite DB (Cell 5).")
    _csv_loaded = False



Looking for CSV: Data\timeseries_2026-04-30.csv
⚠️  CSV not found — will build from SQLite DB (Cell 5).


In [5]:
# %% -- CELL 5 : BUILD TIMESERIES DATAFRAME (RUNS ONLY IF CSV ABSENT) ----------
if not _csv_loaded:

    print("Reading markets table from SQLite ...")
    conn = sqlite3.connect(str(local_path))
    raw = pd.read_sql_query(f"SELECT * FROM {TABLE_NAME}", conn)
    conn.close()
    print(f"   Raw shape: {raw.shape}")

    # -- basic cleaning ----------------------------------------------------------
    raw[SCRAPED_AT_COL] = pd.to_datetime(raw[SCRAPED_AT_COL], errors="coerce")
    bad_ts = raw[SCRAPED_AT_COL].isna().sum()
    if bad_ts:
        print(f"   Dropping {bad_ts} rows with invalid timestamps.")
        raw = raw.dropna(subset=[SCRAPED_AT_COL])

    raw = raw.sort_values(SCRAPED_AT_COL).reset_index(drop=True)

    # Drop configured columns (ignore any that are not actually present)
    drop_present = [c for c in COLS_TO_DROP if c in raw.columns]
    raw = raw.drop(columns=drop_present)
    print(f"   After dropping {len(drop_present)} metadata cols -> {raw.shape[1]} columns remain.")

    # -- keep only rows where question is not null/blank -------------------------
    raw[QUESTION_COL] = raw[QUESTION_COL].astype("string").str.strip()
    raw = raw.dropna(subset=[QUESTION_COL])
    raw = raw[raw[QUESTION_COL] != ""]
    print(f"   Rows with non-null question: {len(raw):,}")

    # -- report unique markets ---------------------------------------------------
    n_markets = raw[QUESTION_COL].nunique()
    n_timestamps = raw[SCRAPED_AT_COL].nunique()
    print(f"   Unique markets    : {n_markets}")
    print(f"   Unique timestamps : {n_timestamps}")

    # -- pivot to wide timeseries ------------------------------------------------
    # Result: MultiIndex columns -> level 0 = variable, level 1 = question
    # Rows  : scraped_at (one row per scrape tick)
    value_cols = [c for c in raw.columns if c not in {SCRAPED_AT_COL, QUESTION_COL}]

    print(f"\nPivoting {len(value_cols)} value columns x {n_markets} markets ...")
    print("   (this may take a minute on large datasets)")

    dup_count = raw.duplicated(subset=[SCRAPED_AT_COL, QUESTION_COL]).sum()
    if dup_count:
        print(f"   Found {dup_count:,} duplicate (timestamp, question) rows; keeping the last row per pair.")

    raw_dedup = raw.drop_duplicates(subset=[SCRAPED_AT_COL, QUESTION_COL], keep="last")

    df = raw_dedup.pivot_table(
        index=SCRAPED_AT_COL,
        columns=QUESTION_COL,
        values=value_cols,
        aggfunc="last",
        observed=False,
    )
    df.index.name = SCRAPED_AT_COL
    df = df.sort_index()

    print("Pivot complete.")
    print(f"   Final shape : {df.shape}  (rows x [variable x market] columns)")
    print(f"   Column levels: {df.columns.names}")

    # -- save to CSV -------------------------------------------------------------
    print(f"\nSaving to {csv_path} ...")
    df.to_csv(csv_path)
    print(f"CSV saved ({csv_path.stat().st_size / 1024 / 1024:.1f} MB).")

Reading markets table from SQLite ...
   Raw shape: (866116, 103)
   After dropping 22 metadata cols -> 81 columns remain.
   Rows with non-null question: 866,116
   Unique markets    : 245
   Unique timestamps : 8698

Pivoting 79 value columns x 245 markets ...
   (this may take a minute on large datasets)
Pivot complete.
   Final shape : (8698, 17923)  (rows x [variable x market] columns)
   Column levels: [None, 'question']

Saving to Data\timeseries_2026-04-30.csv ...
CSV saved (807.1 MB).


In [6]:
# %% ── CELL 6 : SANITY CHECK ──────────────────────────────────────────────────
print("=" * 70)
print("  SANITY CHECK")
print("=" * 70)

# ── overview ──────────────────────────────────────────────────────────────────
n_rows, n_cols     = df.shape
variables          = df.columns.get_level_values(0).unique().tolist()
markets            = df.columns.get_level_values(1).unique().tolist()
first_ts           = df.index.min()
last_ts            = df.index.max()
span_days          = (last_ts - first_ts).total_seconds() / 86400

print(f"\n{'── DIMENSIONS':─<50}")
print(f"  Rows (scraped_at timestamps) : {n_rows:>8,}")
print(f"  Columns total                : {n_cols:>8,}")
print(f"  Unique variables             : {len(variables):>8,}")
print(f"  Unique markets               : {len(markets):>8,}")

print(f"\n{'── TIME RANGE':─<50}")
print(f"  First observation  : {first_ts}")
print(f"  Last  observation  : {last_ts}")
print(f"  Span               : {span_days:.1f} days")

# ── inferred average interval ─────────────────────────────────────────────────
if n_rows > 1:
    deltas         = df.index.to_series().diff().dropna()
    median_gap_min = deltas.median().total_seconds() / 60
    print(f"  Median scrape gap  : {median_gap_min:.1f} min")

# ── per-variable summary across ALL markets ───────────────────────────────────
print(f"\n{'── PER-VARIABLE COVERAGE (across all markets)':─<50}")
rows_fmt = []
for var in variables:
    sub        = df[var]                     # DataFrame: rows=timestamps, cols=markets
    total_cells = sub.size
    non_null    = sub.count().sum()
    fill_pct    = 100 * non_null / total_cells if total_cells else 0
    # first & last non-null value anywhere in this variable
    stacked = sub.stack(future_stack=True).dropna()
    first_val  = stacked.iloc[0]  if len(stacked) else None
    last_val   = stacked.iloc[-1] if len(stacked) else None
    rows_fmt.append({
        "variable"  : var,
        "fill_%"    : round(fill_pct, 1),
        "non_null"  : non_null,
        "first_val" : first_val,
        "last_val"  : last_val,
    })

summary_df = pd.DataFrame(rows_fmt).sort_values("fill_%", ascending=False)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60)
print(summary_df.to_string(index=False))

# ── market list ───────────────────────────────────────────────────────────────
print(f"\n{'── MARKETS IN DATASET':─<50}")
for i, mkt in enumerate(sorted(markets), 1):
    print(f"  {i:>3}. {mkt[:90]}")

print("\n✅ Sanity check complete.")


  SANITY CHECK

── DIMENSIONS─────────────────────────────────────
  Rows (scraped_at timestamps) :    8,698
  Columns total                :   17,923
  Unique variables             :       79
  Unique markets               :      245

── TIME RANGE─────────────────────────────────────
  First observation  : 2026-03-30 15:15:52.586000
  Last  observation  : 2026-04-30 22:01:00.703000
  Span               : 31.3 days
  Median scrape gap  : 5.0 min

── PER-VARIABLE COVERAGE (across all markets)─────
                    variable  fill_%  non_null                                                                                                                                                                                                                                                             first_val                                                                                                                                                                                             

In [7]:
# %% ── 2 OBS PER DAY: outcomes, outcomePrices, spread ─────────────────────────

MARKET    = "Will Bitcoin outperform Gold in 2026?"
VARIABLES = ["outcomes", "outcomePrices", "spread"]

# 1. Slice the variables for this specific market
# Note: Keep the original DatetimeIndex for grouping
market_slice = df.xs(MARKET, axis=1, level=1)[VARIABLES]

# 2. Group by the date and take the first 2 observations of each day
# .tail(2) also works if you prefer the last two observations of the day
daily_sample = market_slice.groupby(market_slice.index.date).head(2)

# 3. Formatting for display
daily_sample.index = daily_sample.index.strftime("%Y-%m-%d %H:%M:%S")

pd.set_option("display.max_rows", 500) 
pd.set_option("display.float_format", "{:.4f}".format)

print(f"Market : {MARKET}")
print(f"Total Daily Samples: {len(daily_sample)}")
print("-" * 30)
print(daily_sample.to_string())

Market : Will Bitcoin outperform Gold in 2026?
Total Daily Samples: 64
------------------------------
                          outcomes       outcomePrices  spread
scraped_at                                                    
2026-03-30 15:15:52  ["Yes", "No"]    ["0.34", "0.66"]  0.0200
2026-03-30 15:21:07  ["Yes", "No"]    ["0.34", "0.66"]  0.0200
2026-03-31 10:35:47  ["Yes", "No"]    ["0.34", "0.66"]  0.0200
2026-03-31 10:40:35  ["Yes", "No"]    ["0.34", "0.66"]  0.0200
2026-04-01 00:00:47  ["Yes", "No"]    ["0.34", "0.66"]  0.0200
2026-04-01 00:05:45  ["Yes", "No"]    ["0.34", "0.66"]  0.0200
2026-04-02 00:00:33  ["Yes", "No"]  ["0.325", "0.675"]  0.0100
2026-04-02 00:05:32  ["Yes", "No"]  ["0.335", "0.665"]  0.0100
2026-04-03 00:00:43  ["Yes", "No"]    ["0.33", "0.67"]  0.0200
2026-04-03 00:05:38  ["Yes", "No"]    ["0.33", "0.67"]  0.0200
2026-04-04 00:00:39  ["Yes", "No"]  ["0.345", "0.655"]  0.0100
2026-04-04 00:05:55  ["Yes", "No"]  ["0.345", "0.655"]  0.0100
2026-04-05 00:00

In [8]:
#how many "questions" do we have in the dataset?
questions = df.columns.get_level_values(1).unique()
print(f"\nTotal unique questions (markets): {len(questions)}")


Total unique questions (markets): 245


In [9]:
# %% -- CELL 10 : TABULAR INTEGRATION (MARKET FILTER) ----------------------------
# Build a model-ready panel by first filtering to gold-related markets.

GOLD_KEYWORDS = [
    "gold", "xauusd", "xau",
]

all_questions = df.columns.get_level_values(1).dropna().unique()
gold_questions = sorted(
    q for q in all_questions
    if isinstance(q, str) and any(kw in q.lower() for kw in GOLD_KEYWORDS)
)

if not gold_questions:
    raise ValueError(
        "No gold-related markets found. Update GOLD_KEYWORDS or inspect market names in Cell 6."
    )

print(f"Gold-related markets found : {len(gold_questions)} / {len(all_questions)}")
for i, q in enumerate(gold_questions, 1):
    print(f"  {i:>3}. {q}")

df_gold_full = df.loc[:, df.columns.get_level_values(1).isin(gold_questions)].copy()
print(f"\nShape after market filter : {df_gold_full.shape}")

Gold-related markets found : 239 / 245
    1. Gold (XAUUSD) Up or Down on April 10?
    2. Gold (XAUUSD) Up or Down on April 13?
    3. Gold (XAUUSD) Up or Down on April 14?
    4. Gold (XAUUSD) Up or Down on April 15?
    5. Gold (XAUUSD) Up or Down on April 16?
    6. Gold (XAUUSD) Up or Down on April 17?
    7. Gold (XAUUSD) Up or Down on April 1?
    8. Gold (XAUUSD) Up or Down on April 20?
    9. Gold (XAUUSD) Up or Down on April 21?
   10. Gold (XAUUSD) Up or Down on April 22?
   11. Gold (XAUUSD) Up or Down on April 23?
   12. Gold (XAUUSD) Up or Down on April 24?
   13. Gold (XAUUSD) Up or Down on April 27?
   14. Gold (XAUUSD) Up or Down on April 28?
   15. Gold (XAUUSD) Up or Down on April 29?
   16. Gold (XAUUSD) Up or Down on April 2?
   17. Gold (XAUUSD) Up or Down on April 30?
   18. Gold (XAUUSD) Up or Down on April 6?
   19. Gold (XAUUSD) Up or Down on April 7?
   20. Gold (XAUUSD) Up or Down on April 8?
   21. Gold (XAUUSD) Up or Down on April 9?
   22. Gold (XAUUSD) U

In [10]:
# %% -- CELL 11 : TABULAR INTEGRATION (NUMERIC FILTER, FLATTEN, SAVE) ------------
# Keep only numeric, time-varying variables and flatten columns for tabular models.

import re
from hashlib import md5

NUMERIC_VARS = [
    "bestAsk",
    "bestBid",
    "lastTradePrice",
    "spread",
    "liquidityClob",
    "volume",
    "volume24hr",
    "volume1wk",
    "volume1mo",
    "oneHourPriceChange",
    "oneDayPriceChange",
    "oneWeekPriceChange",
    "oneMonthPriceChange",
]

available_vars = df_gold_full.columns.get_level_values(0).unique().tolist()
vars_to_keep = [v for v in NUMERIC_VARS if v in available_vars]
vars_dropped = [v for v in NUMERIC_VARS if v not in available_vars]
if vars_dropped:
    print(f"Variables not found in data (skipped): {vars_dropped}")

if not vars_to_keep:
    raise ValueError(
        "None of the configured NUMERIC_VARS are present after market filtering. "
        "Inspect df_gold_full.columns.get_level_values(0).unique()."
    )

df_gold_mi = df_gold_full.loc[:, df_gold_full.columns.get_level_values(0).isin(vars_to_keep)]

def _slugify(text: str, maxlen: int = 55) -> str:
    text = text.lower()[:maxlen]
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")

def _flat_col_name(var: str, market: str) -> str:
    # Add a short stable hash to prevent collisions between similar long question texts.
    suffix = md5(market.encode("utf-8")).hexdigest()[:6]
    return f"{var}__{_slugify(market)}_{suffix}"

df_gold = df_gold_mi.copy()
df_gold.columns = [_flat_col_name(var, mkt) for var, mkt in df_gold_mi.columns]
df_gold.index.name = SCRAPED_AT_COL

print(f"Shape after variable filter + flatten : {df_gold.shape}")
print(
    f"  -> {len(vars_to_keep)} variables x {len(gold_questions)} markets "
    f"= {len(vars_to_keep) * len(gold_questions)} columns (max; less if some combos missing)"
)
print(f"  -> {len(df_gold)} timestamp rows")

print("\nSample column names (first 5):")
for c in df_gold.columns[:5]:
    print(f"  {c}")

coverage = df_gold.notna().mean().sort_values(ascending=False)
print("\nCoverage summary (% non-null per column):")
print(f"  Median fill : {coverage.median() * 100:.1f}%")
print(f"  < 10% fill  : {(coverage < 0.10).sum()} columns")
print(f"  > 80% fill  : {(coverage > 0.80).sum()} columns")

gold_csv_name = f"gold_panel_{file_date}.csv"
gold_csv_path = Path(LOCAL_DATA_DIR) / gold_csv_name
df_gold.to_csv(gold_csv_path)
print(f"\nGold panel saved -> {gold_csv_path}")
print(f"File size : {gold_csv_path.stat().st_size / 1024:.1f} KB")

Shape after variable filter + flatten : (8698, 2781)
  -> 13 variables x 239 markets = 3107 columns (max; less if some combos missing)
  -> 8698 timestamp rows

Sample column names (first 5):
  bestAsk__gold_xauusd_up_or_down_on_april_10_4ba50c
  bestAsk__gold_xauusd_up_or_down_on_april_13_4d2a29
  bestAsk__gold_xauusd_up_or_down_on_april_14_62ec42
  bestAsk__gold_xauusd_up_or_down_on_april_15_098dc9
  bestAsk__gold_xauusd_up_or_down_on_april_16_d0076d

Coverage summary (% non-null per column):
  Median fill : 16.8%
  < 10% fill  : 1089 columns
  > 80% fill  : 680 columns

Gold panel saved -> Data\gold_panel_2026-04-30.csv
File size : 88462.6 KB
